In [1]:
! pip install kagglehub[pandas-datasets]

## 1. Целевая переменная

В рамках ВКР по разработке конвейера данных для платёжной информации в качестве основной задачи мною было выбрано **обнаружение аномальных транзакций**, которые могут указывать на мошеннические операции.

Поскольку исходный датасет не содержит меток мошенничества, целевая переменная `is_anomaly` была создана на основе статистических и временных критериев:

1. **Статистические аномалии** — транзакции с суммой, значительно отклоняющейся от среднего по счёту (|z-score| > 2.5)
2. **Временные аномалии** — транзакции, следующие слишком быстро после предыдущей (интервал < 300 секунд)

Оба критерия при этом выбраны на основе проведенного ранее EDA

In [2]:
import kagglehub
import pandas as pd
import os
import datetime as dt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, f1_score

dataset_path = kagglehub.dataset_download("mdhossanr/financial-transactions-dataset-for-analysis")
file_path = os.path.join(dataset_path, "Financial Transactions.csv")
df = pd.read_csv(file_path, encoding='utf-8', engine='python', encoding_errors='ignore')

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['hour'] = df['Timestamp'].dt.hour
df['date'] = df['Timestamp'].dt.date
df['month'] = df['Timestamp'].dt.month
df['year'] = df['Timestamp'].dt.year
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['weekend'] = df['day_of_week'].apply(lambda x: 1 if x > 4 else 0)
df['time'] = df['Timestamp'].dt.time

# Статистические характеристики по счетам
df['avg_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
df['std_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('std')

# Обработка нулевого стандартного отклонения
std_safe = df['std_amount_per_account'].replace(0, 1)
df['amount_zscore'] = (df['TransactionAmount'] - df['avg_amount_per_account']) / std_safe
df['amount_zscore'] = df['amount_zscore'].fillna(0)

# Аномалии по времени
df_sorted = df.sort_values(['AccountID', 'Timestamp'])
df_sorted['time_since_last'] = df_sorted.groupby('AccountID')['Timestamp'].diff().dt.total_seconds()
df['time_since_last'] = df_sorted['time_since_last'].fillna(999999)

# Первоначальные критерии
df['is_anomaly'] = ((df['amount_zscore'].abs() > 2.5) | (df['time_since_last'] < 300)).astype(int)

print(f'Число аномалий в датасете: {df['is_anomaly'].sum()}')

Using Colab cache for faster access to the 'financial-transactions-dataset-for-analysis' dataset.
Число аномалий в датасете: 1


Количество аномалий в датасете оказалось слишком малым (1 аномалия на весь датасет), это приведет к невозможности использования моделей ML для дальнейшей работы. Поэтому необходимо увеличить число аномалий синтетически

***Замечание:*** В реальных данных такой исход невозможен, потому на них этап с увеличением числа аномалий необходимо пропустить.

In [3]:
non_anomaly_indices = df[df['is_anomaly'] == 0].index.tolist()
needed = 100 - df['is_anomaly'].sum()
np.random.seed(42)
new_anomaly_indices = np.random.choice(non_anomaly_indices, needed, replace=False)

df.loc[new_anomaly_indices, 'TransactionAmount'] = df.loc[new_anomaly_indices, 'TransactionAmount'] * 3
df.loc[new_anomaly_indices, 'is_anomaly'] = 1

df['avg_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('mean')
df['std_amount_per_account'] = df.groupby('AccountID')['TransactionAmount'].transform('std')
std_safe = df['std_amount_per_account'].replace(0, 1)
df['amount_zscore'] = (df['TransactionAmount'] - df['avg_amount_per_account']) / std_safe
df['amount_zscore'] = df['amount_zscore'].fillna(0)

print(f'Итоговое число аномалий: {df['is_anomaly'].sum()}')
print(f'Доля аномалий: {df['is_anomaly'].mean():.2%}')

Итоговое число аномалий: 100
Доля аномалий: 0.27%


1) Для baseline были выбраны три модели разной сложности:

- DummyClassifier - базовый уровень для сравнения, показывает минимальный порог качества
- Logistic Regression - интерпретируемая линейная модель, позволяет оценить линейную разделимость данных
- Random Forest - ансамблевый метод, устойчив к выбросам и дисбалансу классов, хорошо работает с табличными данными

2) Признаки для модели (отобранные на основеании проведенного ранее EDA)

- TransactionAmount (сумма транзакции) — ключевой показатель для выявления аномалий
- AccountBalance (баланс после транзакции) — отражает финансовое состояние счёта
- hour	(час транзакции) — позволяет выявить нехарактерное время операций
- day_of_week	(день недели) — выявление активности в выходные
- weekend	(флаг выходного дня) — агрегированный признак
- amount_zscore	(отклонение суммы от среднего по счёту) — ключевой признак аномалии

3) Метрики оценки

- Recall	(доля реальных аномалий, правильно обнаруженных моделью) - пропуск мошенничества может привести к финансовым потерям. Наиболее приоритетная метрика
- Precision	(доля предсказанных аномалий, которые действительно являются аномалиями) - важен для минимизации ложных срабатываний. Наименее приоритетная метрика
- F1-score	(гармоническое среднее precision и recall) - позволяет оценить баланс между полнотой и точностью. Вторая по приоритету метрика
- ROC-AUC	(способность модели разделять классы независимо от порога классификации).

In [4]:
features = ['TransactionAmount', 'AccountBalance', 'hour', 'day_of_week', 'weekend', 'amount_zscore']

X = df[features]
y = df['is_anomaly']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Размер обучающей выборки: {len(X_train)}')
print(f'Размер тестовой выборки: {len(X_test)}')
print(f'Аномалии в обучающей: {y_train.sum()}')
print(f'Аномалии в тестовой: {y_test.sum()}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Dummy (most_frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100)
}

results = {}
for name, model in models.items():
    print(f'\nОбучение {name}')

    if name == 'Logistic Regression':
        X_train_use = X_train_scaled
        X_test_use = X_test_scaled
    else:
        X_train_use = X_train
        X_test_use = X_test

    model.fit(X_train_use, y_train)
    y_pred = model.predict(X_test_use)

    y_proba = model.predict_proba(X_test_use)
    y_proba = y_proba[:, 1]
    roc_auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc
    }

    print(classification_report(y_test, y_pred, zero_division=0))
    print(f'ROC-AUC: {roc_auc:.4f}')

print('\nПроверка моделей на переобучение\n')
for name, model in models.items():
    if name == 'Logistic Regression':
        X_train_use = X_train_scaled
        X_test_use = X_test_scaled
    else:
        X_train_use = X_train
        X_test_use = X_test

    train_pred = model.predict(X_train_use)
    test_pred = model.predict(X_test_use)
    train_f1 = f1_score(y_train, train_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    print(f'\n{name}:')
    print(f'Train F1: {train_f1:.4f}')
    print(f'Test F1: {test_f1:.4f}')
    print(f'Разница: {train_f1 - test_f1:.4f}')

Размер обучающей выборки: 29933
Размер тестовой выборки: 7484
Аномалии в обучающей: 80
Аномалии в тестовой: 20

Обучение Dummy (most_frequent)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7464
           1       0.00      0.00      0.00        20

    accuracy                           1.00      7484
   macro avg       0.50      0.50      0.50      7484
weighted avg       0.99      1.00      1.00      7484

ROC-AUC: 0.5000

Обучение Logistic Regression
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7464
           1       1.00      0.45      0.62        20

    accuracy                           1.00      7484
   macro avg       1.00      0.72      0.81      7484
weighted avg       1.00      1.00      1.00      7484

ROC-AUC: 0.8078

Обучение Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7464
        

По результатам обучения можно заметить, что две более сложные модели показали признаки переобучения:

1) Logistic Regression: train F1 = 0.74, test F1 = 0.62
2) Random Forest: train F1 = 1.00, test F1 = 0.79

Следовательно необходимо улучшить оба варианта, для этого буду применять GridSearch с целью поиска наилучших параметров каждой модели.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 4, 8],
    'class_weight': ['balanced']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(f'Лучшие параметры Random Forest: {rf_grid.best_params_}')
print(f'Лучший F1 на кросс-валидации: {rf_grid.best_score_:.4f}')
print(f'Test F1 (оптимизированный RF): {f1_score(y_test, y_pred_rf):.4f}')

lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'class_weight': ['balanced']
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    lr_params,
    cv=5,
    scoring='f1'
)

lr_grid.fit(X_train_scaled, y_train)
best_lr = lr_grid.best_estimator_
y_pred_lr = best_lr.predict(X_test_scaled)

print(f'\nЛучшие параметры Logistic Regression: {lr_grid.best_params_}')
print(f'Лучший F1 на кросс-валидации: {lr_grid.best_score_:.4f}')
print(f'Test F1 (оптимизированный LR): {f1_score(y_test, y_pred_lr):.4f}')

Лучшие параметры Random Forest: {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}
Лучший F1 на кросс-валидации: 0.8343
Test F1 (оптимизированный RF): 0.7879

Лучшие параметры Logistic Regression: {'C': 0.01, 'class_weight': 'balanced', 'penalty': 'l2'}
Лучший F1 на кросс-валидации: 0.0578
Test F1 (оптимизированный LR): 0.0537


По результатам можно заметить, что Logistic Regression показывает сильное падение качества, при этом переобучение в Random Forest устранено, что позволяет выбрать Random Forest в качестве основной модели.

## Выводы по этапу.

1) Задача обнаружения аномалий успешно формализована как бинарная классификация с созданной целевой переменной на основе статистических и временных критериев.
2) Random Forest показал наилучшие результаты среди рассмотренных моделей, достигнув F1-score 0.79 при отсутствии переобучения.
3) Ключевые признаки для модели:
- amount_zscore (отклонение от среднего по счёту)
- TransactionAmount (сумма транзакции)
- временные признаки (час, день недели)
4) Логистическая регрессия оказалась неэффективной из-за нелинейного характера зависимостей и сильного дисбаланса классов.